# Results analysis (Student 4)
Training curves, results table, error breakdown and error analysis for the report.
**Rule:** test-split numbers come only from the final, dev-selected checkpoint.

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

# finds the repo root and makes src/ importable
repo = Path.cwd()
if not (repo / "configs").is_dir():
    repo = repo.parent
sys.path.insert(0, str(repo / "src"))

from conformer_asr.evaluation.metrics import edit_counts, normalize
from conformer_asr.evaluation.visualization import (load_scalars, plot_error_breakdown,
                                                    plot_loss_curves, plot_scalar)

# settings: need to change for real run
RUN = "fake_run" # Noah's experiment_name
TAG = "best" # checkpoint tag used by evaluate.py
SELECTED_STEP = 4500 # step of the selected checkpoint

log_dir = repo / "outputs" / "tensorboard" / RUN
metrics_dir = repo / "outputs" / "metrics"
predictions_dir = repo / "outputs" / "predictions"
fig_dir = repo / "docs" / "report" / "figures"
fig_dir.mkdir(parents=True, exist_ok=True)
print("repo:", repo)
print("log dir exists:", log_dir.is_dir())

## 1. Training curves

In [ ]:
series = load_scalars(log_dir)
print("Logged scalars:", sorted(series))

plot_loss_curves(log_dir, fig_dir / "loss_curves", smoothing=0.9, selected_step=SELECTED_STEP)
plot_scalar(log_dir, "train/learning_rate", fig_dir / "learning_rate", "Learning rate",
            selected_step=SELECTED_STEP)
plot_scalar(log_dir, "val/wer", fig_dir / "val_wer", "dev-clean WER", selected_step=SELECTED_STEP)

for name in ("loss_curves", "learning_rate", "val_wer"):
    display(Image(filename=str(fig_dir / f"{name}.png"), width=500))

## 2. Results table

In [ ]:
SPLITS = ["dev-clean", "dev-other", "test-clean", "test-other"]
PAPER = {"test-clean": 2.7, "test-other": 6.3}

metrics = {}
for split in SPLITS:
    path = metrics_dir / f"{split}_{TAG}.json"
    if path.exists():
        metrics[split] = json.loads(path.read_text())
    else:
        print(f"missing: {path.name}")

rows = []
for split in SPLITS:
    m = metrics.get(split)
    rows.append({
        "split": split,
        "WER %": round(100 * m["wer"], 2) if m else None,
        "CER %": round(100 * m["cer"], 2) if m else None,
        "S": m["word_error_counts"]["substitutions"] if m else None,
        "D": m["word_error_counts"]["deletions"] if m else None,
        "I": m["word_error_counts"]["insertions"] if m else None,
        "ref words": m["reference_words"] if m else None,
        "utterances": m["utterances"] if m else None,
        "paper WER %": PAPER.get(split),
    })
table = pd.DataFrame(rows).set_index("split")
table

In [ ]:
def latex_cell(split, key):
    m = metrics.get(split)
    return f"{100 * m[key]:.2f}" if m else "--"

print("Ours, WER (greedy, no LM) & " + " & ".join(latex_cell(s, "wer") for s in SPLITS) + r" \\")
print("Ours, CER (greedy, no LM) & " + " & ".join(latex_cell(s, "cer") for s in SPLITS) + r" \\")

## 3. Error breakdown

In [ ]:
available = [metrics[s] for s in SPLITS if s in metrics]
plot_error_breakdown(available, fig_dir / "error_breakdown")
display(Image(filename=str(fig_dir / "error_breakdown.png"), width=450))

## 4. Error analysis

In [ ]:
records = []
for split in SPLITS:
    path = predictions_dir / f"{split}_{TAG}.jsonl"
    if not path.exists():
        continue
    for line in path.read_text(encoding="utf-8").splitlines():
        r = json.loads(line)
        ref, hyp = normalize(r["reference"]).split(), normalize(r["prediction"]).split()
        counts = edit_counts(ref, hyp)
        r.update(split=split, ref_words=len(ref), errors=sum(counts.values()), **counts)
        records.append(r)

df = pd.DataFrame(records)
df["utt_wer"] = df.errors / df.ref_words.clip(lower=1)
print(len(df), "utterances loaded")
df.groupby("split")[["errors", "ref_words"]].sum().assign(
    WER=lambda g: (100 * g.errors / g.ref_words).round(2)
)

In [ ]:
# how many utterances are recognized perfectly
exact = df.groupby("split").apply(lambda g: (g.errors == 0).mean() * 100).round(1)
print("Exactly correct (% of utterances):\n", exact, "\n")

# is WER worse for short or long utterances
df["length"] = pd.cut(df.ref_words, [0, 5, 10, 20, 40, 1000],
                      labels=["1-5", "6-20", "11-20", "21-40", "40+"])
by_length = (df.groupby(["split", "length"], observed=True)[["errors", "ref_words"]].sum()
             .assign(WER=lambda g: (100 * g.errors / g.ref_words).round(2))["WER"]
             .unstack())
by_length

In [ ]:
import difflib
from collections import Counter

def substitution_pairs(reference, prediction):
    ref, hyp = normalize(reference).split(), normalize(prediction).split()
    pairs = []
    matcher = difflib.SequenceMatcher(a=ref, b=hyp, autojunk=False)
    for op, i1, i2, j1, j2 in matcher.get_opcodes():
        if op == "replace" and (i2 - i1) == (j2 - j1): # same length replaced block
            pairs += zip(ref[i1:i2], hyp[j1:j2])
    return pairs

counter = Counter(p for r in records for p in substitution_pairs(r["reference"], r["prediction"]))
pd.DataFrame([(f"{a} -> {b}", n) for (a, b), n in counter.most_common(15)],
             columns=["reference -> hypothesis", "count"])

In [ ]:
pd.set_option("display.max_colwidth", 200)
clean = df[df.split == "test-clean"] if (df.split == "test-clean").any() else df

print("Correct examples:")
display(clean[clean.errors == 0].sample(min(3, (clean.errors == 0).sum()), random_state=0)
        [["utterance_id", "reference", "prediction"]])

print("Incorrect examples (most errors first):")
display(clean[clean.errors > 0].sort_values("errors", ascending=False).head(5)
        [["utterance_id", "reference", "prediction", "substitutions", "deletions", "insertions"]])